# Fetching Historical Market Data Using ALPHA VANTAGE API

- Starting by pulling earnings history dates from the "Magnificent Seven" Tech Stocks

- AAPL, MSFT, AMZN, GOOGL, META, NVDA, TSLA

## Alpha Vantage Methodology
- Enter Ticker, find all historical earnings dates (function = EARNINGS)
- Enter Ticker & Date range, find all news headlines + URL's
(function = NEWS_SENTIMENT)
- Enter Ticker & Date range, retrieve candlesticks
(function = TIME_SERIES_DAILY)


In [1]:
library(httr2)
library(jsonlite)
library(dplyr)
library(tibble)
library(tidyverse)
library(rvest)
library(httr2)
library(googledrive)
library(googlesheets4)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.1.6
✔ ggplot2   4.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter()  masks stats::filter()
✖ purrr::flatten() masks jsonlite::flatten()
✖ dplyr::lag()     masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘rvest’


The following object is masked from ‘package:readr’:

    guess_encoding



Attaching package: ‘googlesheets4’


The following objects are masked from ‘package:googledrive’:

    request_generate, request_make




In [2]:
av_earnings_dates_RAG <- function(symbol, api_key, since = as.Date("2022-01-01")) {
  stopifnot(is.character(symbol), length(symbol) == 1, nchar(symbol) > 0)
  stopifnot(is.character(api_key), length(api_key) == 1, nchar(api_key) > 0)

  # Normalize 'since' to Date
  if (!inherits(since, "Date")) {
    since <- tryCatch(as.Date(since), error = function(e) NA)
  }
  if (is.na(since)) stop("'since' must be coercible to Date (e.g., '2005-01-01').")

  base_url <- "https://www.alphavantage.co/query"
  res <- httr::GET(
    url = base_url,
    query = list('function' = "EARNINGS", symbol = symbol, apikey = api_key),
    httr::user_agent("av_earnings_dates_r/1.0")
  )

  if (httr::http_error(res)) {
    status <- httr::status_code(res)
    stop(sprintf("HTTP error from Alpha Vantage (status %s).", status))
  }

  txt <- httr::content(res, as = "text", encoding = "UTF-8")
  parsed <- jsonlite::fromJSON(txt, simplifyVector = TRUE)

  # Error Handling
  if ("Note" %in% names(parsed)) stop(sprintf("Alpha Vantage note: %s", parsed$Note))
  if ("Information" %in% names(parsed)) stop(sprintf("Alpha Vantage information: %s", parsed$Information))
  if ("Error Message" %in% names(parsed)) stop(sprintf("Alpha Vantage error: %s", parsed$`Error Message`))

  q <- parsed[["quarterlyEarnings"]]
  if (is.null(q) || NROW(q) == 0) {
    # CHANGE 1: Define earnings_id as character(0) here
    return(data.frame(
      earnings_id = character(0),
      symbol = character(0),
      reportedDate = as.Date(character(0)),
      fiscalDateEnding = as.Date(character(0)),
      reportedEPS = numeric(0),
      surprise = numeric(0),
      surprisePercentage = numeric(0),
      stringsAsFactors = FALSE
    ))
  }

  # Coerce types safely
  to_num <- function(x) suppressWarnings(as.numeric(x))
  to_date <- function(x) suppressWarnings(as.Date(x))

  q$reportedDate <- to_date(q$reportedDate)
  q$fiscalDateEnding <- to_date(q$fiscalDateEnding)

  if ("reportedEPS" %in% names(q)) q$reportedEPS <- to_num(q$reportedEPS)
  if ("surprise" %in% names(q)) q$surprise <- to_num(q$surprise)
  if ("surprisePercentage" %in% names(q)) q$surprisePercentage <- to_num(q$surprisePercentage)

  keep <- (!is.na(q$reportedDate) & q$reportedDate >= since) |
          (is.na(q$reportedDate) & !is.na(q$fiscalDateEnding) & q$fiscalDateEnding >= since)
  q <- q[keep, , drop = FALSE]

  q$symbol <- symbol
  q <- q[order(q$reportedDate, decreasing = TRUE, na.last = TRUE), , drop = FALSE]

  # CHANGE 2: Create the String ID
  # We use sprintf "%02d" to add a leading zero (01, 02) to keep sorting clean up to 99 rows.
  # If you want just "AAPL1", change this line to: paste0(symbol, seq_len(nrow(q)))
  q$earnings_id <- paste0(symbol, sprintf("%02d", seq_len(nrow(q))))

  # Reorder columns
  cols <- c("earnings_id", "symbol", "reportedDate", "fiscalDateEnding", "reportedEPS", "surprise", "surprisePercentage")
  cols <- intersect(cols, names(q))
  q <- q[, unique(c(cols, names(q))), drop = FALSE]

  rownames(q) <- NULL
  q
}

In [ ]:
api_key <- "DA7HRX4GDRJMC1X9"

aapl_earnings <- av_earnings_dates_RAG("AAPL", api_key, since = as.Date("2022-01-01"))
head(aapl_earnings)



,earnings_id,symbol,reportedDate,fiscalDateEnding,reportedEPS,surprise,surprisePercentage,estimatedEPS,reportTime
,<chr>,<chr>,<date>,<date>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,AAPL01,AAPL,2025-10-30,2025-09-30,1.85,0.09,5.1136,1.76,post-market
2,AAPL02,AAPL,2025-07-31,2025-06-30,1.57,0.14,9.7902,1.43,post-market
3,AAPL03,AAPL,2025-05-01,2025-03-31,1.65,0.03,1.8519,1.62,post-market
4,AAPL04,AAPL,2025-01-30,2024-12-31,2.40,0.06,2.5641,2.34,post-market
5,AAPL05,AAPL,2024-10-31,2024-09-30,0.97,0.02,2.1053,0.95,post-market
6,AAPL06,AAPL,2024-08-01,2024-06-30,1.40,0.06,4.4776,1.34,post-market


In [3]:
# Tickers must be character strings
dow <- c("MMM","AXP","AMGN","AMZN","AAPL","BA","CAT","CVX","CSCO","KO","DIS","GS","HD","HON","IBM","JNJ","JPM","MCD","MRK","MSFT","NKE","NVDA","PG","CRM","SHW","TRV","UNH","VZ","V","WMT"
)

# Your Alpha Vantage key (or set once via Sys.setenv)
api_key <- "DA7HRX4GDRJMC1X9"

# We'll store each ticker's table in a named list
results <- vector("list", length(dow))
names(results) <- dow

for (i in seq_along(dow)) {
  sym <- dow[i]
  df  <- av_earnings_dates_RAG(
    symbol = sym,
    api_key = api_key,
    since = as.Date("2022-03-01")
  )
  results[[i]] <- df

  # Peek at the first rows for each ticker
  cat("\n", sym, "— first 6 rows:\n")
  print(utils::head(df))
}

# Combine all tickers into one data frame
library(dplyr)
all_earnings <- bind_rows(results)
all_earnings # Lets look at all results
# aapl_earnings <- results[["AAPL"]]
# head(aapl_earnings)



 MMM — first 6 rows:
  earnings_id symbol reportedDate fiscalDateEnding reportedEPS surprise
1       MMM01    MMM   2025-10-21       2025-09-30        2.19     0.12
2       MMM02    MMM   2025-07-18       2025-06-30        2.16     0.15
3       MMM03    MMM   2025-04-22       2025-03-31        1.88     0.11
4       MMM04    MMM   2025-01-21       2024-12-31        1.68     0.14
5       MMM05    MMM   2024-10-22       2024-09-30        1.98     0.08
6       MMM06    MMM   2024-07-26       2024-06-30        1.93     0.25
  surprisePercentage estimatedEPS reportTime
1             5.7971         2.07 pre-market
2             7.4627         2.01 pre-market
3             6.2147         1.77 pre-market
4             9.0909         1.54 pre-market
5             4.2105          1.9 pre-market
6            14.8810         1.68 pre-market

 AXP — first 6 rows:
  earnings_id symbol reportedDate fiscalDateEnding reportedEPS surprise
1       AXP01    AXP   2025-10-17       2025-09-30        4.14   

earnings_id,symbol,reportedDate,fiscalDateEnding,reportedEPS,surprise,surprisePercentage,estimatedEPS,reportTime
<chr>,<chr>,<date>,<date>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
MMM01,MMM,2025-10-21,2025-09-30,2.19,0.12,5.7971,2.07,pre-market
MMM02,MMM,2025-07-18,2025-06-30,2.16,0.15,7.4627,2.01,pre-market
MMM03,MMM,2025-04-22,2025-03-31,1.88,0.11,6.2147,1.77,pre-market
MMM04,MMM,2025-01-21,2024-12-31,1.68,0.14,9.0909,1.54,pre-market
MMM05,MMM,2024-10-22,2024-09-30,1.98,0.08,4.2105,1.9,pre-market
MMM06,MMM,2024-07-26,2024-06-30,1.93,0.25,14.8810,1.68,pre-market
MMM07,MMM,2024-04-30,2024-03-31,2.39,0.28,13.2701,2.11,pre-market
MMM08,MMM,2024-01-23,2023-12-31,2.42,0.11,4.7619,2.31,pre-market
MMM09,MMM,2023-10-24,2023-09-30,2.68,0.33,14.0426,2.35,pre-market


## Earnings date releases secured
- we can now move on to pulling the News Based upon each given date!

In [ ]:
# helper: infix to replace NULL with default
`%||%` <- function(x, y) if (is.null(x)) y else x

get_av_news_articles <- function(
  reported_date,
  tickers,
  api_key,
  report_time,              # e.g. "pre-market", "after-market", etc.
  earnings_id = NA,         # attaches to every returned row
  topics = NULL,
  sort = c("LATEST", "EARLIEST", "RELEVANCE"),
  limit = 100,              # per-day limit (respect your plan; AV max can be higher on premium)
  sleep_between_calls = 0   # seconds to sleep between each API call (per day)
) {
  # ---- 0) Validate inputs ----
  if (missing(reported_date) || is.null(reported_date)) {
    stop("reported_date is required (YYYY-MM-DD).")
  }
  if (missing(tickers) || is.null(tickers)) {
    stop("tickers is required (e.g., 'AAPL' or c('AAPL','CRYPTO:BTC')).")
  }
  if (missing(api_key) || is.null(api_key)) {
    stop("api_key is required.")
  }
  if (missing(report_time) || is.null(report_time)) {
    stop("report_time is required (e.g. 'pre-market', 'after-market').")
  }

  sort  <- match.arg(sort)
  limit <- as.integer(limit)
  if (is.na(limit) || limit <= 0L) stop("limit must be a positive integer.")
  # if you know your plan allows more, you can bump this up
  if (limit > 1000L) {
    warning("Alpha Vantage NEWS_SENTIMENT max is 1000; capping at 1000.")
    limit <- 1000L
  }

  # Normalize reported_date to Date
  rd <- tryCatch(as.Date(reported_date), error = function(e) NA)
  if (is.na(rd)) {
    stop("reported_date must be coercible to Date (e.g., '2024-09-10').")
  }

  # ---- 1) Helper to parse time_published (Alpha Vantage returns UTC) ----
  parse_time_published <- function(x) {
    if (length(x) == 0) return(as.POSIXct(character()))
    n <- nchar(x)
    has_seconds <- n >= 15
    out <- rep(as.POSIXct(NA, tz = "UTC"), length(x))
    if (any(has_seconds)) {
      out[has_seconds] <- as.POSIXct(strptime(x[has_seconds], "%Y%m%dT%H%M%S", tz = "UTC"))
    }
    if (any(!has_seconds)) {
      out[!has_seconds] <- as.POSIXct(strptime(x[!has_seconds], "%Y%m%dT%H%M", tz = "UTC"))
    }
    out
  }

  # ---- 2) Determine earnings event time for PRE vs POST classification ----
  rt_lower <- tolower(report_time %||% "")
  earnings_hour <- if (grepl("pre", rt_lower, fixed = TRUE)) 7L else 16L

  earnings_event_time <- as.POSIXct(
    sprintf("%s %02d:00:00", format(rd, "%Y-%m-%d"), earnings_hour),
    tz = "UTC"
  )

  # ---- 3) Inner function to call AV for a single day ----
  call_for_one_day <- function(day) {
    t_from <- paste0(format(day, "%Y%m%d"), "T0000")
    t_to   <- paste0(format(day, "%Y%m%d"), "T2359")

    q <- list(
      `function` = "NEWS_SENTIMENT",
      tickers    = paste(tickers, collapse = ","),
      time_from  = t_from,
      time_to    = t_to,
      sort       = sort,   # use requested sort
      limit      = limit,  # per-day limit
      apikey     = api_key
    )
    if (!is.null(topics) && length(topics)) {
      q$topics <- paste(topics, collapse = ",")
    }

    resp <- httr::GET("https://www.alphavantage.co/query", query = q)
    if (httr::http_error(resp)) {
      stop(sprintf("HTTP error %s: %s",
                   httr::status_code(resp),
                   httr::http_status(resp)$message))
    }

    txt <- httr::content(resp, as = "text", encoding = "UTF-8")
    js  <- jsonlite::fromJSON(txt, simplifyVector = FALSE)

    # handle API notes/errors
    if (!is.null(js$Note)) {
      # js$Note is often rate-limit or usage info; treat as fatal so you notice it
      stop(sprintf("Alpha Vantage note: %s", js$Note))
    }

    if (!is.null(js$Information)) {
      info_msg <- js$Information
      # "No articles found" should NOT kill the loop – just skip this day
      if (grepl("No articles found", info_msg, fixed = TRUE)) {
        return(NULL)
      } else {
        stop(sprintf("Alpha Vantage info: %s", info_msg))
      }
    }

    if (is.null(js$feed) || length(js$feed) == 0) {
      return(NULL)  # no articles for this day
    }

    feed_df <- jsonlite::fromJSON(txt, flatten = TRUE)$feed
    tp <- parse_time_published(feed_df$time_published %||% NA_character_)

    # PRE vs POST relative to earnings_event_time (same for all days)
    rel_flag <- ifelse(tp < earnings_event_time, "PRE", "POST")

    tibble::tibble(
      earnings_id         = as.character(earnings_id),  # <-- make sure it's character
      ticker              = tickers,
      reported_date       = rd,
      report_time         = report_time,
      earnings_event_time = earnings_event_time,
      time_from           = t_from,
      time_to             = t_to,
      title               = feed_df$title %||% NA_character_,
      url                 = feed_df$url %||% NA_character_,
      source              = feed_df$source %||% NA_character_,
      source_domain       = feed_df$source_domain %||% NA_character_,
      time_published      = tp,
      relative_to_earnings = rel_flag
    )
  }

  # ---- 4) Seven calls: rd-3, rd-2, rd-1, rd, rd+1, rd+2, rd+3 ----
  day_seq <- rd + (-3:3)

  day_results <- vector("list", length(day_seq))
  for (k in seq_along(day_seq)) {
    if (k > 1 && sleep_between_calls > 0) {
      Sys.sleep(sleep_between_calls)
    }
    day_results[[k]] <- call_for_one_day(day_seq[k])
  }

  # drop NULL days
  day_results <- day_results[!vapply(day_results, is.null, logical(1))]

  if (length(day_results) == 0L) {
    # no articles across all 7 days
    return(tibble::tibble(
      earnings_id         = character(),           # <-- character(), not numeric()
      ticker              = character(),
      reported_date       = as.Date(character()),
      report_time         = character(),
      earnings_event_time = as.POSIXct(character()),
      time_from           = character(),
      time_to             = character(),
      title               = character(),
      url                 = character(),
      source              = character(),
      source_domain       = character(),
      time_published      = as.POSIXct(character()),
      relative_to_earnings = character()
    ))
  }

  out <- do.call(rbind, day_results)

  # ---- 5) Global sort by recency (latest first) ----
  out <- out[order(out$time_published, decreasing = TRUE), , drop = FALSE]

  out
}


In [ ]:
articles <- get_av_news_articles(
  reported_date = "2022-09-10",
  tickers       = "MSFT",
  report_time   = "post-market",
  api_key       = "x",         # your key
  earnings_id   = 1,           # <-- this will be carried on every row
  sort          = "LATEST",
  limit         = 100
)

dplyr::count(articles, earnings_id)


earnings_id,n
<chr>,<int>
1,262


In [ ]:
results <- vector("list", nrow(all_earnings))

for (i in seq_len(nrow(all_earnings))) {
  row <- all_earnings[i, ]

  df <- tryCatch(
    get_av_news_articles(
      reported_date       = row$reportedDate,
      tickers             = row$symbol,
      api_key             = "DA7HRX4GDRJMC1X9",
      report_time         = row$reportTime,
      earnings_id         = row$earnings_id,
      sort                = "LATEST",
      limit               = 100,
      sleep_between_calls = 1   # <- 1 second between each of the 7 day-calls
    ),
    error = function(e) {
      message("Error on ", row$earnings_id, " (", row$symbol, "): ", e$message)
      NULL
    }
  )

  results[[i]] <- df
}

all_news <- dplyr::bind_rows(results)


Error on JPM01 (JPM): Alpha Vantage info: Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.

Error on JPM02 (JPM): Alpha Vantage info: Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.

Error on JPM03 (JPM): Alpha Vantage info: Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.

Error on JPM04 (JPM): Alpha Vantage info: Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.

Error on JPM05 (JPM): Alpha Vantage info: Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.

Error on JPM06 (JPM): Alpha Vantage info: Invalid inputs. Please refer to the API documentation https://www.alphavantage.co/documentation#newsapi and try again.

Error on JPM07 (JPM): Alpha 

In [ ]:
all_news

earnings_id,ticker,reported_date,report_time,earnings_event_time,time_from,time_to,title,url,source,source_domain,time_published,relative_to_earnings
<chr>,<chr>,<date>,<chr>,<dttm>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dttm>,<chr>
MMM01,MMM,2025-10-21,pre-market,2025-10-21 07:00:00,20251022T0000,20251022T2359,"3M's Modest Results Win Analyst Praise, But PFAS Litigation Risk Looms - 3M ( NYSE:MMM )",https://www.benzinga.com/analyst-stock-ratings/price-target/25/10/48363125/3ms-modest-results-win-analyst-praise-but-pfas-litigation-risk-looms,Benzinga,www.benzinga.com,2025-10-22 18:23:10,POST
MMM01,MMM,2025-10-21,pre-market,2025-10-21 07:00:00,20251022T0000,20251022T2359,3M Analysts Boost Their Forecasts After Upbeat Q3 Earnings - 3M ( NYSE:MMM ),https://www.benzinga.com/news/25/10/48357844/3m-analysts-boost-their-forecasts-after-upbeat-q3-earnings,Benzinga,www.benzinga.com,2025-10-22 15:37:51,POST
MMM01,MMM,2025-10-21,pre-market,2025-10-21 07:00:00,20251022T0000,20251022T2359,"Stock Market News for Oct 22, 2025",https://www.zacks.com/stock/news/2773953/stock-market-news-for-oct-22-2025,Zacks Commentary,www.zacks.com,2025-10-22 13:20:00,POST
MMM01,MMM,2025-10-21,pre-market,2025-10-21 07:00:00,20251021T0000,20251021T2359,"3M's Q3 Earnings & Revenues Surpass Estimates, Increase Y/Y",https://www.zacks.com/stock/news/2773455/3ms-q3-earnings-revenues-surpass-estimates-increase-yy,Zacks Commentary,www.zacks.com,2025-10-21 16:51:00,POST
MMM01,MMM,2025-10-21,pre-market,2025-10-21 07:00:00,20251021T0000,20251021T2359,3M Gears Up For Q3 Print; Here Are The Recent Forecast Changes From Wall Street's Most Accurate Analysts - 3M ( NYSE:MMM ),https://www.benzinga.com/analyst-stock-ratings/price-target/25/10/48320786/3m-gears-up-for-q3-print-here-are-the-recent-forecast-changes-from-wall-street,Benzinga,www.benzinga.com,2025-10-21 07:36:23,POST
MMM01,MMM,2025-10-21,pre-market,2025-10-21 07:00:00,20251020T0000,20251020T2359,"Top Stocks With Earnings This Week: Tesla, Netflix, RTX and More - Tesla ( NASDAQ:TSLA )",https://www.benzinga.com/trading-ideas/movers/25/10/48299369/retail-investors-top-stocks-with-earnings-this-week-tesla-netflix-intel-and-more,Benzinga,www.benzinga.com,2025-10-20 11:53:01,PRE
MMM02,MMM,2025-07-18,pre-market,2025-07-18 07:00:00,20250721T0000,20250721T2359,NEOG Investor Notice: Robbins LLP Reminds Investors of the Class Action Lawsuit Against Neogen Corporation - Neogen ( NASDAQ:NEOG ),https://www.benzinga.com/pressreleases/25/07/g46533372/neog-investor-notice-robbins-llp-reminds-investors-of-the-class-action-lawsuit-against-neogen-corp,Benzinga,www.benzinga.com,2025-07-21 21:03:15,POST
MMM02,MMM,2025-07-18,pre-market,2025-07-18 07:00:00,20250721T0000,20250721T2359,"Zacks Industry Outlook Highlights Honeywell, 3M, Carlisle Companies and Federal Signal",https://www.zacks.com/stock/news/2598799/zacks-industry-outlook-highlights-honeywell-3m-carlisle-companies-and-federal-signal,Zacks Commentary,www.zacks.com,2025-07-21 09:45:00,POST
MMM02,MMM,2025-07-18,pre-market,2025-07-18 07:00:00,20250721T0000,20250721T2359,"Dow Sinks Over 100 Points As Sentiment Slips, Yet Fear & Greed Index Still Flashes 'Extreme Greed' - American Express ( NYSE:AXP ) , Domino's Pizza ( NASDAQ:DPZ )",https://www.benzinga.com/markets/equities/25/07/46513114/dow-sinks-over-100-points-as-sentiment-slips-yet-fear-greed-index-still-flashes-extreme-greed,Benzinga,www.benzinga.com,2025-07-21 08:46:59,POST


In [ ]:
library(dplyr)

news_capped <- all_news %>%
  group_by(earnings_id, relative_to_earnings) %>%   # PRE and POST separately
  arrange(desc(time_published), .by_group = TRUE) %>%  # latest first within each group
  slice_head(n = 25) %>%                             # cap at 25 per (earnings_id, PRE/POST)
  ungroup()

news_capped


earnings_id,ticker,reported_date,report_time,earnings_event_time,time_from,time_to,title,url,source,source_domain,time_published,relative_to_earnings
<chr>,<chr>,<date>,<chr>,<dttm>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dttm>,<chr>
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,Is This Nevada-Based Company a Strong Play for Growth-Oriented Portfolios?,https://www.fool.com/investing/2025/11/02/is-this-nevada-based-company-a-strong-play-for-gro/,Motley Fool,www.fool.com,2025-11-02 23:05:00,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,Apple Gears Up For A Pivotal Year With Major Product Releases And Challenges - Apple ( NASDAQ:AAPL ),https://www.benzinga.com/markets/tech/25/11/48581363/apple-gears-up-for-a-pivotal-year-with-major-product-releases-and-challenges,Benzinga,www.benzinga.com,2025-11-02 20:37:04,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,1 Secret to Finding the Next Apple That Is Hiding in Plain Sight,https://www.fool.com/investing/2025/11/02/1-secret-to-finding-the-next-apple-that-is-hiding/,Motley Fool,www.fool.com,2025-11-02 18:05:00,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,25% of Warren Buffett's $315 Billion Portfolio Is Invested in 2 Artificial Intelligence ( AI ) Stocks,https://www.fool.com/investing/2025/11/02/25-of-warren-buffetts-315-billion-portfolio-is-inv/,Motley Fool,www.fool.com,2025-11-02 17:00:00,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,The Single Best AI Stock: Could It Surge 148% by 2030?,https://www.fool.com/investing/2025/11/02/single-best-ai-stock-could-it-surge-148-by-2030/,Motley Fool,www.fool.com,2025-11-02 15:45:00,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,The Vanguard Information Technology ETF ( VGT ) Offers Broader Tech Diversification Than the Technology Select Sector SPDR Fund ( XLK ),https://www.fool.com/coverage/etfs/2025/11/02/vgt-offers-broader-tech-diversification-than-xlk/,Motley Fool,www.fool.com,2025-11-02 14:19:05,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,Better Artificial Intelligence ETF: Technology Select Sector SPDR Fund vs. Roundhill Generative AI & Technology ETF,https://www.fool.com/coverage/etfs/2025/11/02/better-artificial-intelligence-etf-technology-select-sector-spdr-fund-vs-roundhill-generative-ai-and-technology-etf/,Motley Fool,www.fool.com,2025-11-02 14:00:05,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,Vanguard VOO ETF Offers Lower Costs and Stronger Growth Than IWM,https://www.fool.com/coverage/etfs/2025/11/02/vanguard-voo-etf-offers-lower-costs-and-stronger-growth-than-iwm/,Motley Fool,www.fool.com,2025-11-02 13:01:04,POST
AAPL01,AAPL,2025-10-30,post-market,2025-10-30 16:00:00,20251102T0000,20251102T2359,"Is Nvidia Still a Safe Bet if the ""AI Bubble"" Deflates?",https://www.fool.com/investing/2025/11/02/is-nvidia-still-safe-bet-if-ai-bubble-deflate/,Motley Fool,www.fool.com,2025-11-02 10:42:00,POST


In [ ]:
library(httr)
library(rvest)
library(xml2)
library(dplyr)

scrape_article_text <- function(
  url,
  selectors = NULL,
  timeout_sec = 10, # Reduced timeout to fail faster on bad links
  user_agent_str = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
  min_chars_fallback = 200
) {
  # Safety check for empty or NA URLs
  if (is.na(url) || !nzchar(url)) return(NA_character_)

  # Wrap entire logic in tryCatch to handle HTTP errors/Timeouts gracefully
  tryCatch({
    # 1) Fetch HTML
    resp <- httr::GET(
      url,
      httr::user_agent(user_agent_str),
      httr::timeout(timeout_sec)
      # accept("text/html") removed to be more permissive
    )

    # If status is not 200 (OK), return NA immediately
    if (httr::status_code(resp) != 200) return(NA_character_)

    doc <- xml2::read_html(httr::content(resp, as = "raw"))

    # 2) Remove non-content noise
    remove_xpaths <- c(
      "//script", "//style", "//noscript", "//header", "//footer",
      "//nav", "//form", "//aside", "//figure", "//figcaption",
      "//svg", "//iframe", "//img", "//picture", "//button",
      "//video", "//audio", "//div[contains(@class, 'share')]",
      "//div[contains(@class, 'related')]"
    )
    for (xp in remove_xpaths) {
      xml2::xml_remove(xml2::xml_find_all(doc, xp))
    }

    # 3) Candidate selectors
    default_selectors <- c(
      'article [itemprop="articleBody"]', 'article .article-body',
      'main [itemprop="articleBody"]', 'main .article-body',
      'div[itemprop="articleBody"]', '.article-body',
      '.article__content', '.post-content', '.entry-content',
      '#article-body', '#articleBody', 'article', 'main article',
      'section[itemprop="articleBody"]', 'section.article-body',
      'article .content'
    )
    cand_selectors <- if (is.null(selectors)) default_selectors else c(selectors, default_selectors)

    # 4) Helper functions
    score_node <- function(node) {
      ps <- rvest::html_elements(node, "p")
      if (length(ps) == 0) return(list(nchar = 0L, text = character(0)))
      txt <- rvest::html_text2(ps)
      txt <- txt[nzchar(txt)]
      list(nchar = nchar(paste(txt, collapse = " ")), text = txt)
    }

    pick_best_from_nodes <- function(nodeset) {
      best_len <- -1L
      best_txt <- NULL
      for (nd in nodeset) {
        sc <- score_node(nd)
        if (sc$nchar > best_len) {
          best_len <- sc$nchar
          best_txt <- sc$text
        }
      }
      list(nchar = best_len, text = best_txt)
    }

    # 5) Try CSS selectors
    best <- list(nchar = -1L, text = character(0))
    for (sel in cand_selectors) {
      nodes <- rvest::html_elements(doc, sel)
      if (length(nodes) == 0) next
      candidate <- pick_best_from_nodes(nodes)
      if (candidate$nchar > best$nchar) best <- candidate
    }

    # 6) Fallback: largest text block
    if (best$nchar < min_chars_fallback) {
      candidates <- rvest::html_elements(
        doc,
        xpath = '//*[self::article or self::div or self::section or self::main][count(.//p) > 1]'
      )
      if (length(candidates) > 0) {
        candidate <- pick_best_from_nodes(candidates)
        if (candidate$nchar > best$nchar) best <- candidate
      }
    }

    # 7) Last-resort fallback
    if (best$nchar < min_chars_fallback) {
      body_ps <- rvest::html_elements(doc, "body p")
      if (length(body_ps) > 0) {
        txt <- rvest::html_text2(body_ps)
        txt <- txt[nzchar(txt)]
        best <- list(nchar = nchar(paste(txt, collapse = " ")), text = txt)
      }
    }

    # 8) Normalize
    if (length(best$text) == 0) return(NA_character_)

    norm <- function(x) {
      x <- gsub("[ \t]+", " ", x)
      trimws(x)
    }
    best$text <- vapply(best$text, norm, character(1))
    best$text <- best$text[nzchar(best$text)]
    paste(best$text, collapse = "\n\n")

  }, error = function(e) {
    # On ANY error, return NA so the loop continues
    return(NA_character_)
  })
}



Attaching package: ‘xml2’


The following object is masked from ‘package:httr2’:

    url_parse




In [ ]:
url <- "https://www.benzinga.com/analyst-stock-ratings/price-target/25/10/48320786/3m-gears-up-for-q3-print-here-are-the-recent-forecast-changes-from-wall-street"
article_text <- scrape_article_text(url)
cat(substr(article_text, 1, 80000))

3M Company (NYSE:MMM) will release earnings results for the third quarter, before the opening bell on Tuesday, Oct. 21.

Analysts expect the Saint Paul, Minnesota-based company to report quarterly earnings at $2.07 per share, up from $1.98 per share in the year-ago period. The consensus estimate for 3M's quarterly revenue is $6.25 billion, compared to $6.07 billion a year earlier, according to data from Benzinga Pro.

On July 18, 3M reported better-than-expected second-quarter financial results and raised its FY25 adjusted EPS guidance.

Shares of 3M gained 1.4% to close at $154.78 on Monday.

Benzinga readers can access the latest analyst ratings on the Analyst Stock Ratings page. Readers can sort by stock ticker, company name, analyst firm, rating change or other variables.

Let's have a look at how Benzinga's most-accurate analysts have rated the company in the recent period.

Considering buying GEV stock? Here’s what analysts think:

Read This Next:

Photo via Shutterstock

© 2025 

In [ ]:
news_capped <- read_csv("/news_capped.csv")

# Save news_capped as CSV
#write.csv(news_capped, "all_earnings.csv", row.names = FALSE)

Rows: 15394 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (8): earnings_id, ticker, report_time, title, url, source, source_domai...
dttm (4): earnings_event_time, time_from, time_to, time_published
date (1): reported_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
library(dplyr)

# 1. Extract unique URLs
unique_url_df <- news_capped %>%
  distinct(url) %>%
  filter(!is.na(url), url != "")

cat("Total rows in news_capped:", nrow(news_capped), "\n")
cat("Unique URLs to scrape:", nrow(unique_url_df), "\n")

# 2. Initialize column
unique_url_df$scraped_text <- NA_character_

total   <- nrow(unique_url_df)
start_t <- Sys.time()

# --- SAFETY ADDITION 1: Define a temporary save file ---
temp_file <- "news_scrape_checkpoint.rds"

if (total == 0) {
  warning("No valid URLs found in news_capped.")
} else {

  # --- SAFETY ADDITION 2: Resume capability (Optional but helpful) ---
  if (file.exists(temp_file)) {
    message("Found checkpoint file! Resuming...")
    unique_url_df <- readRDS(temp_file)
    # Start at the first NA row
    first_na <- min(which(is.na(unique_url_df$scraped_text)))
    # If all done, start from total + 1 to skip loop
    start_index <- if (is.finite(first_na)) first_na else (total + 1)
  } else {
    start_index <- 1
  }

  cat("Starting scrape at", format(start_t), "from index", start_index, "\n")

  # Loop
  for (i in start_index:total) {
    # If we are already past the end (because of resume), break
    if (i > total) break

    current_url <- unique_url_df$url[i]

    # Progress logging
    if (i %% 50 == 0 || i == 1 || i == total) {
      elapsed <- as.numeric(difftime(Sys.time(), start_t, units = "secs"))
      pct     <- (i / total) * 100
      cat(sprintf(
        "[%s] Processing %d of %d (%.1f%%) — elapsed: %.1fs\n",
        format(Sys.time(), "%H:%M:%S"),
        i, total, pct, elapsed
      ))

      # --- SAFETY ADDITION 3: Save to disk every 50-100 rows ---
      # This ensures if R crashes, you only lose the last few minutes of work.
      saveRDS(unique_url_df, temp_file)
    }

    # Scrape this URL
    # CRITICAL: Ensure scrape_article_text() has the tryCatch block inside it!
    txt <- scrape_article_text(current_url)
    unique_url_df$scraped_text[i] <- txt

    # Politeness sleep: ~0.25s on average
    # This is fast/aggressive. If you get 429/403 errors, increase this.
    pause <- runif(1, 0.15, 0.4)
    Sys.sleep(pause)
  }

  end_t <- Sys.time()
  cat("Finished scrape at", format(end_t), "\n")
  cat("Total elapsed:",
      round(as.numeric(difftime(end_t, start_t, units = "mins")), 2),
      "minutes\n")
}

# 4. Join scraped text back to the capped news set
all_news_final <- news_capped %>%
  left_join(unique_url_df, by = "url")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




ERROR: Error: object 'news_capped' not found


In [ ]:
all_earnings
write.csv(all_earnings, "/all_earnings.csv", row.names = FALSE)

ERROR: Error: object 'all_earnings' not found


This function must:

intake row of earnings
parse the inputted date:
Creates two variables....
45 days prior (to provided date), 45 days post (to provided date)

uses two seperate API calls on daily ohlcv bars using alpha vantage: INSERT SAMPLES CODE


calculates for each alpha vantage api call.
Price Range: Calculated by finding the difference between the high and low price. This indicates the price volatility during the period.

Price Change: The difference between the closing price and the opening price, which shows the net price movement.
Volume-based ratios

Price-to-Volume: A ratio of the price to the volume can provide insights into the strength of a price movement


will create six new rows.

price_range_prior
price_change_prior
avg_volume_prior
price_range_post
price_change_post
avg_volume_post


In [8]:
fetch_alpha_vantage_data <- function(symbol, api_key) {
  # 1. Construct the API URL
  # ADDED: &outputsize=full to get 20 years of data, not just 100 days
  base_url <- "https://www.alphavantage.co/query"
  query_params <- paste0(
    "function=TIME_SERIES_DAILY",
    "&symbol=", symbol,
    "&apikey=", api_key,
    "&datatype=csv",
    "&outputsize=full"
  )
  full_url <- paste0(base_url, "?", query_params)

  # 2. Try to fetch the data
  tryCatch({
    stock_data <- read_csv(full_url, show_col_types = FALSE)

    # 3. Check for API Errors (Rate Limits usually return a specific text in the CSV)
    # We check the first column name for specific error signatures
    first_col <- names(stock_data)[1]

    if (nrow(stock_data) == 0 ||
        grepl("Error", first_col) ||
        grepl("Information", first_col)) {
        # "Information" usually means "Thank you for using... please upgrade" (Rate Limit)

      warning(paste("API Error for:", symbol, "- Limit Reached or Bad Symbol"))
      return(NULL)
    }

    # 4. Standardize Columns
    colnames(stock_data) <- tolower(colnames(stock_data))
    stock_data$timestamp <- as.Date(stock_data$timestamp)

    return(stock_data)

  }, error = function(e) {
    warning(paste("Network error for:", symbol, "-", e$message))
    return(NULL)
  })
}

In [14]:
library(dplyr)
library(readr)

fetch_alpha_vantage_data <- function(symbol, api_key) {
  # 1. CHANGE: Use TIME_SERIES_DAILY_ADJUSTED to handle splits automatically
  base_url <- "https://www.alphavantage.co/query"
  query_params <- paste0(
    "function=TIME_SERIES_DAILY_ADJUSTED", # <--- CRITICAL FIX 1
    "&symbol=", symbol,
    "&apikey=", api_key,
    "&datatype=csv",
    "&outputsize=full"
  )
  full_url <- paste0(base_url, "?", query_params)

  tryCatch({
    stock_data <- read_csv(full_url, show_col_types = FALSE)

    first_col <- names(stock_data)[1]
    if (nrow(stock_data) == 0 || grepl("Error", first_col) || grepl("Information", first_col)) {
      warning(paste("API Error for:", symbol))
      return(NULL)
    }

    colnames(stock_data) <- tolower(colnames(stock_data))
    stock_data$timestamp <- as.Date(stock_data$timestamp)

    # 2. CHANGE: Explicitly sort Oldest -> Newest so 'head' is always the start date
    stock_data <- stock_data %>% arrange(timestamp) # <--- CRITICAL FIX 2

    return(stock_data)

  }, error = function(e) {
    warning(paste("Network error for:", symbol, "-", e$message))
    return(NULL)
  })
}

process_earnings_data <- function(earnings_df, api_key, sleep_sec = 1.5) {

  results <- earnings_df %>%
    mutate(
      price_range_prior = NA_real_,
      price_change_prior = NA_real_,
      avg_volume_prior   = NA_real_,
      price_range_post   = NA_real_,
      price_change_post  = NA_real_,
      avg_volume_post    = NA_real_
    )

  stock_cache <- list()
  total_rows <- nrow(results)

  for (i in 1:total_rows) {
    row_symbol <- results$symbol[i]
    # Ensure date format is correct
    report_date <- as.Date(results$reportedDate[i])

    # --- Check Cache ---
    if (!row_symbol %in% names(stock_cache)) {
      message(paste("Fetching:", row_symbol, "(", i, "of", total_rows, ")"))
      stock_data <- fetch_alpha_vantage_data(row_symbol, api_key)
      stock_cache[[row_symbol]] <- stock_data
      if (i < total_rows) Sys.sleep(sleep_sec)
    } else {
      stock_data <- stock_cache[[row_symbol]]
    }

    if (is.null(stock_data)) next

    # --- Date Windows (No Look Ahead) ---
    # Prior: 45 days before to 3 days before
    start_prior <- report_date - 45
    end_prior   <- report_date - 3

    # Post: 3 days after to 45 days after (Day 4 to Day 45)
    start_post <- report_date + 3
    end_post   <- report_date + 45

    # --- Subset ---
    data_prior <- stock_data %>% filter(timestamp >= start_prior & timestamp < end_prior)
    data_post  <- stock_data %>% filter(timestamp >= start_post & timestamp <= end_post)

    # --- Calculate Prior ---
    if (nrow(data_prior) > 0) {
      results$price_range_prior[i] <- max(data_prior$high, na.rm = TRUE) - min(data_prior$low, na.rm = TRUE)

      # 3. CHANGE: Use ADJUSTED CLOSE for returns.
      # Because we sorted by timestamp earlier: head=Start, tail=End
      start_price <- head(data_prior$adjusted_close, 1)
      end_price   <- tail(data_prior$adjusted_close, 1)

      results$price_change_prior[i] <- (end_price - start_price) / start_price * 100
      results$avg_volume_prior[i] <- mean(data_prior$volume, na.rm = TRUE)
    }

    # --- Calculate Post ---
    if (nrow(data_post) > 0) {
      results$price_range_post[i] <- max(data_post$high, na.rm = TRUE) - min(data_post$low, na.rm = TRUE)

      # 3. CHANGE: Use ADJUSTED CLOSE for returns
      start_price <- head(data_post$adjusted_close, 1)
      end_price   <- tail(data_post$adjusted_close, 1)

      results$price_change_post[i] <- (end_price - start_price) / start_price * 100
      results$avg_volume_post[i] <- mean(data_post$volume, na.rm = TRUE)
    }
  }

  return(results)
}

In [15]:
# 1. Load libraries and the function we created
library(dplyr)
library(readr)
AV_API_KEY <- "DA7HRX4GDRJMC1X9"
final_results <- process_earnings_data(all_earnings, api_key = AV_API_KEY)

# 4. View Results
final_results

Fetching: MMM ( 1 of 450 )

Fetching: AXP ( 16 of 450 )

Fetching: AMGN ( 31 of 450 )

Fetching: AMZN ( 46 of 450 )

Fetching: AAPL ( 61 of 450 )

Fetching: BA ( 76 of 450 )

Fetching: CAT ( 91 of 450 )

Fetching: CVX ( 106 of 450 )

Fetching: CSCO ( 121 of 450 )

Fetching: KO ( 136 of 450 )

Fetching: DIS ( 151 of 450 )

Fetching: GS ( 166 of 450 )

Fetching: HD ( 181 of 450 )

Fetching: HON ( 196 of 450 )

Fetching: IBM ( 211 of 450 )

Fetching: JNJ ( 226 of 450 )

Fetching: JPM ( 241 of 450 )

Fetching: MCD ( 256 of 450 )

Fetching: MRK ( 271 of 450 )

Fetching: MSFT ( 286 of 450 )

Fetching: NKE ( 301 of 450 )

Fetching: NVDA ( 316 of 450 )

Fetching: PG ( 331 of 450 )

Fetching: CRM ( 346 of 450 )

Fetching: SHW ( 361 of 450 )

Fetching: TRV ( 376 of 450 )

Fetching: UNH ( 391 of 450 )

Fetching: VZ ( 406 of 450 )

Fetching: V ( 421 of 450 )

Fetching: WMT ( 436 of 450 )



earnings_id,symbol,reportedDate,fiscalDateEnding,reportedEPS,surprise,surprisePercentage,estimatedEPS,reportTime,price_range_prior,price_change_prior,avg_volume_prior,price_range_post,price_change_post,avg_volume_post
<chr>,<chr>,<date>,<date>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MMM01,MMM,2025-10-21,2025-09-30,2.19,0.12,5.7971,2.07,pre-market,13.6450,-0.8831169,2751600,13.5000,1.6158058,2970021
MMM02,MMM,2025-07-18,2025-06-30,2.16,0.15,7.4627,2.01,pre-market,17.9000,6.6023088,2939875,16.2200,2.2500335,3400825
MMM03,MMM,2025-04-22,2025-03-31,1.88,0.11,6.2147,1.77,pre-market,33.0229,-11.7937949,4483595,19.8800,6.4808595,2962535
MMM04,MMM,2025-01-21,2024-12-31,1.68,0.14,9.0909,1.54,pre-market,16.2900,7.2553046,3396343,15.5700,-1.6137292,4055070
MMM05,MMM,2024-10-22,2024-09-30,1.98,0.08,4.2105,1.9,pre-market,13.6000,5.2000623,3238388,11.0800,7.2768784,3109696
MMM06,MMM,2024-07-26,2024-06-30,1.93,0.25,14.8810,1.68,pre-market,7.2600,3.6024477,3555190,15.6800,3.1815295,4019773
MMM07,MMM,2024-04-30,2024-03-31,2.39,0.28,13.2701,2.11,pre-market,20.9600,3.8287160,4795860,11.0300,4.5906103,4771044
MMM08,MMM,2024-01-23,2023-12-31,2.42,0.11,4.7619,2.31,pre-market,10.9500,4.4590927,3280993,6.4235,-0.5783465,3954939
MMM09,MMM,2023-10-24,2023-09-30,2.68,0.33,14.0426,2.35,pre-market,22.4700,-19.7794049,3792965,17.9400,19.9612090,3255915


In [16]:
write.csv(final_results, "earnings_with_financials2.csv", row.names = FALSE)

## Notes from Professor Steffen
In an embedding space which of stock tickers are closest to STOCK TICKER(s)


Every skill word embedded, thus finding competitors and and excuted a sentiment analysis which encompass all industry news and related stories.

Analyze:


Control variable (Price analysis 60 days before earnings)


|| Change From News|| (news sentiment analysis after earnings)


new Curve (Price analysis 60 days post earnings)


## That is not working.
- Okay it looks like scraping is going to be a little bit illegal

- Lets try use a news api, https://developer.morningstar.com/direct-web-services/documentation/direct-web-services/about

- Morning Star is through the boston college library but is not directly through a terminal.

- I have also gotten in touch with the CSOM librarian for information about the bloomberg terminal. This will be difficult though because I will want to replicate this project after this class.

- Creating and pulling that much data from the terminal and sending it over this laptop could possibly pose issues as well
